# QSITE 2026: Quantum Circuit Compilation - Demo Notebook
**Team / Participant:** Abhay Suresh
**Track:** Computational Track

This notebook accompanies our 3-5 minute presentation and demonstrates the core contributions of our solver:
1. **The Flaws of the Baseline:** Identity placement and greedy routing.
2. **Our Solution Architecture:** Simulated Annealing Placement, Lookahead SABRE Routing, Bidirectional Refinement, and SWAP Decomposition.
3. **Results & Impact:** Dramatic reduction in SWAP operations and circuit depth.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp(prefix="mplcfg-"))

# Ensure starter_kit is in path
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "starter_kit").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "Computational Track" / "starter_kit").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "Computational Track"))

import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import HTML, display

from starter_kit import (
    BENCHMARKS,
    animate_layers,
    animate_routing,
    baseline_solve,
    benchmark_stats,
    build_hardware_graph,
    draw_hardware,
    schedule_layers_ordered,
    score_summary,
)

# Import our custom highly optimized solver
from solver import solve

plt.style.use("seaborn-v0_8-whitegrid")
GRAPH = build_hardware_graph()
print("Setup complete. Hardware graph loaded.")

## Slide 1: Overview & Hardware Topology

Our target is a 20-qubit heavy-hex-style graph. Abstract quantum algorithms assume all-to-all connectivity, but our hardware is constrained.
We must find an optimal logical-to-physical placement and route operations with minimal SWAPs.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
draw_hardware(GRAPH, ax=ax, title="20-Qubit Heavy-Hex Hardware Graph")
plt.tight_layout()
plt.show()

## Slide 2: Explaining the Baseline's Flaws
For circuits like `ghz_star` (a hub-and-spoke logic), the central "hub" qubit should ideally be placed on a central physical node with high degree (e.g. 3).
The baseline's **Identity Placement** blindly puts logical qubit 0 on physical qubit 0 (a corner, degree-2 node).
It also uses greedy shortest-path routing, ignoring future gate dependencies.

In [ ]:
test_program = BENCHMARKS["ghz_star"]

baseline_placement, baseline_routed = baseline_solve(test_program, GRAPH)
bl_score = score_summary(test_program, GRAPH, baseline_placement, baseline_routed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

draw_hardware(GRAPH, placement=baseline_placement, ax=ax1, title="Baseline (Identity) Placement")

# Previewing our solver's placement
our_placement, our_routed = solve(test_program, GRAPH)
our_score = score_summary(test_program, GRAPH, our_placement, our_routed)

draw_hardware(GRAPH, placement=our_placement, ax=ax2, title="Our Solver (SA + Degree-Matching) Placement")
plt.tight_layout()
plt.show()

print(f"Baseline Hub (L0) is on degree {GRAPH.degree(baseline_placement[0])} node.")
print(f"Our Hub (L0) is on degree {GRAPH.degree(our_placement[0])} node.")

## Slide 3: Our Solution Architecture in Action
Our four-stage pipeline:
1. **Smart Placement:** Degree-matching followed by **Simulated Annealing** distance minimization.
2. **SABRE Routing:** Configurable lookahead window (e.g., 20) with exponential decay on future gates.
3. **Bidirectional Refinement:** Reversing the circuit and using backward-routing layout as the forward seed.
4. **Post-Pass Decomposition:** Eliminating algebraically redundant $SWAP \cdot SWAP = I$ chains and reducing 3-cycles.

Let's watch our architecture efficiently route a challenging benchmark!

In [ ]:
print(f"Original program length: {len(test_program)} operations")
print(f"Our routed program length: {len(our_routed)} operations")
print(f"Number of inserted SWAPs: {our_score['swap_count']}")

# Animate the highly-optimized routing sequence
anim = animate_routing(our_placement, our_routed, GRAPH, interval=600)
display(HTML(anim.to_jshtml()))
plt.close('all')

## Slide 4: Results & Impact
By deploying a multi-seed ensemble across different lookahead windows and layout heuristics, our solver achieves a **70.7%** reduction in total score across the 6 benchmarks. Let's compute the results live.

In [ ]:
print(f"{'benchmark':15s}  {'baseline':10s}  {'ours':7s}  {'delta':7s}")
print("-" * 45)

total_baseline = 0
total_ours = 0

for name, prog in BENCHMARKS.items():
    # Baseline
    pl_bl, rt_bl = baseline_solve(prog, GRAPH)
    score_bl = score_summary(prog, GRAPH, pl_bl, rt_bl)["score"]

    # Ours
    pl_opt, rt_opt = solve(prog, GRAPH)
    score_opt = score_summary(prog, GRAPH, pl_opt, rt_opt)["score"]

    total_baseline += score_bl
    total_ours += score_opt

    delta = score_opt - score_bl

    print(f"{name:15s}  {score_bl:5.1f}       {score_opt:5.1f}    {delta:5.1f}")

print("-" * 45)
print(f"{'TOTAL':15s}  {total_baseline:5.1f}       {total_ours:5.1f}    {total_ours - total_baseline:5.1f}")
print(f"\nImprovement: {-(total_ours - total_baseline) / total_baseline * 100:.1f}%!")


## Slide 5: Conclusion
This demo confirms:
1. **Topological awareness** in placement eliminates immediate bottlenecks.
2. **Lookahead heuristics** in routing prevent dead-end convergence.
3. **Decompositions** seamlessly reduce execution depth.

Thank you!